In [ ]:
import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

from xgboost import XGBClassifier

from features import build_features



df = pd.read_csv('data/train_apps.csv')

y = df['target_value']
X = build_features(df)



preprocessor = ColumnTransformer(
    transformers=[
        (
            'onehot',
            OneHotEncoder(
                sparse_output=False,
                handle_unknown='ignore'
            ),
            ['cb_rate']
        ),
        (
            'target',
            TargetEncoder(
                target_type='binary',
                smooth=10,
                random_state=42
            ),
            ['fl_adminarea']
        )
    ],
    remainder='passthrough'
)


model = XGBClassifier(
    objective='binary:logistic',
    max_depth=4,
    learning_rate=0.1,
    n_estimators=100,
    alpha=10,
    random_state=42,
    eval_metric='auc',
    n_jobs=-1
)


pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])



cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    pipe,
    X,
    y,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

print('ROC-AUC по фолдам:', scores)
print('Средний ROC-AUC:', scores.mean())
print('Стандартное отклонение:', scores.std())
pipe.fit(X, y)
joblib.dump(pipe, 'models/model.joblib')
print('Модель сохранена!')